# Earth terrain and land cover


Select FABDEM tiles by their actual footprints, covering the processing halo and interpolation guard. Reproject elevation to separate 30 m UTM grids, derive physical terrain descriptors, and aggregate WorldCover binary indicators into class fractions on those same grids. Source data remain untouched.

The initial natural-ground screening mask is configurable, not a declaration of a valid lunar analog. Height is retained in source meters; relative elevation removes only the regional median. Cross-planet absolute elevations have different reference surfaces. Local elevation standard deviation is a scale-dependent roughness descriptor, not the SDEM VRM product. No training patches or train/test splits are created here.


Run cells in order manually. Raw inputs are read-only. Each processing run writes to a new directory under `data/processed/`; only a `provenance.json` with `status: complete` indicates successful completion. See the [notebook index](../README.md) for environment setup and dependencies.


In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project root or a folder under notebooks")
sys.path.insert(0, str(ROOT))
print("Project:", ROOT)


In [ ]:
import json
import math
import re
import shutil
import zipfile

import numpy as np
import rasterio
from notebooks.shared.processing import (
    AREAS,
    Run,
    landcover_fractions,
    make_grid,
    mosaic_elevation,
    pad_grid,
    roi_mask,
    select_covering_tiles,
    terrain,
    warp,
    write_raster,
)
from rasterio.crs import CRS
from rasterio.warp import transform_bounds

SELECTED_AREAS = list(AREAS)  # Set ["lanzarote"] for a smaller first run.
SPACING_M = 30
ROUGHNESS_WINDOW = 11  # 330 m at the default spacing; complete windows only.
HALO_CELLS = max(1, ROUGHNESS_WINDOW // 2)
INTERPOLATION_GUARD_CELLS = 2
MIN_BARE_FRACTION = 0.8  # Exploratory screening parameter; class 60 includes sparse vegetation.
WORLDCOVER_TILES = {"northern_arizona": "N33W114", "lanzarote": "N27W015", "atacama_san_pedro": "S24W069"}
config = {"areas": {name: AREAS[name] for name in SELECTED_AREAS}, "spacing_m": SPACING_M,
          "roughness_window_cells": ROUGHNESS_WINDOW, "minimum_bare_sparse_fraction": MIN_BARE_FRACTION,
          "elevation_resampling": "bilinear", "class_aggregation": "area-weighted binary indicators",
          "coverage_threshold": 0.99, "terrain_halo_cells": HALO_CELLS,
          "interpolation_guard_cells": INTERPOLATION_GUARD_CELLS,
          "tile_selection": "actual footprints cover padded processing grid plus interpolation guard",
          "relative_elevation_reference": "median elevation inside requested ROI"}
assert SPACING_M >= 30 and 0 <= MIN_BARE_FRACTION <= 1


## Preflight
Inspect ZIP indexes and candidate raster headers. Select actual tile footprints covering the padded processing grid plus an interpolation guard, and require complete footprint coverage. No full source tiles are extracted until the processing cell. Source metadata and archive hashes are retained with each new run.

In [ ]:
archives = sorted((ROOT / "data/raw/earth/fabdem/archives").glob("*.zip"))
member_index = {}
for archive in archives:
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            if item.filename.lower().endswith(".tif"):
                name = Path(item.filename).name
                if name in member_index:
                    raise ValueError(f"Duplicate FABDEM member: {name}")
                member_index[name] = (archive, item.filename, item.CRC, item.file_size)
selected_tiles, cover_paths = {}, {}
output_grids, processing_grids = {}, {}
footprint_cache = {}
selection_bounds = {}
for name in SELECTED_AREAS:
    area = AREAS[name]
    projected = transform_bounds("EPSG:4326", area["crs"], *area["bbox"], densify_pts=41)
    grid = make_grid(projected, area["crs"], SPACING_M)
    work_grid = pad_grid(grid, HALO_CELLS)
    guard_grid = pad_grid(work_grid, INTERPOLATION_GUARD_CELLS)
    guard_bounds = rasterio.transform.array_bounds(guard_grid.height, guard_grid.width, guard_grid.transform)
    geographic = transform_bounds(area["crs"], "EPSG:4326", *guard_bounds, densify_pts=41)
    west, south, east, north = geographic
    candidates = {}
    # A one-degree search margin discovers neighboring names; final selection uses actual headers.
    for tile, (archive, member, _, _) in member_index.items():
        match = re.fullmatch(r"([NS])(\d{2})([EW])(\d{3})_FABDEM_V1-2.tif", tile)
        if match is None:
            continue
        lat = int(match[2]) * (1 if match[1] == "N" else -1)
        lon = int(match[4]) * (1 if match[3] == "E" else -1)
        if not (math.floor(south)-1 <= lat <= math.ceil(north) and math.floor(west)-1 <= lon <= math.ceil(east)):
            continue
        if tile not in footprint_cache:
            with rasterio.open(f"zip://{archive.resolve().as_posix()}!{member}") as src:
                if src.crs != CRS.from_epsg(4326):
                    raise ValueError(f"Unexpected FABDEM CRS: {tile}")
                footprint_cache[tile] = tuple(src.bounds)
        candidates[tile] = footprint_cache[tile]
    required = select_covering_tiles(candidates, geographic)
    selected_tiles[name] = required
    output_grids[name], processing_grids[name] = grid, work_grid
    selection_bounds[name] = geographic
    matches = list((ROOT / "data/raw/earth/worldcover/v200/2021/map").glob(f"*_{WORLDCOVER_TILES[name]}_Map.tif"))
    if len(matches) != 1:
        raise ValueError(f"Expected one WorldCover tile for {name}")
    cover_paths[name] = matches[0]
    print(name, required, matches[0].name)

config["selected_tiles"] = selected_tiles
config["source_selection_bounds_lonlat"] = selection_bounds


In [ ]:
with Run(ROOT, "earth_terrain_landcover", config, "preprocessing/02_earth_terrain_landcover.ipynb") as run:
    summaries = []
    for name in SELECTED_AREAS:
        area = AREAS[name]
        grid = output_grids[name]
        work_grid = processing_grids[name]
        core = (slice(HALO_CELLS, -HALO_CELLS), slice(HALO_CELLS, -HALO_CELLS))
        inside = roi_mask(grid, area["bbox"])
        extracted = []
        source_metadata = []
        for filename in selected_tiles[name]:
            archive, member, crc, size = member_index[filename]
            run.input(archive)
            target = run.path(f"earth/{name}/extracted/{filename}")
            with zipfile.ZipFile(archive) as bundle, bundle.open(member) as source, target.open("wb") as dst:
                shutil.copyfileobj(source, dst, 1024 * 1024)  # ZipExtFile checks member CRC at EOF.
            assert target.stat().st_size == size
            with rasterio.open(target) as src:
                source_metadata.append({"filename": filename, "crc32": crc, "crs": str(src.crs),
                                        "resolution": src.res, "nodata": src.nodata, "units": src.units,
                                        "bounds": list(src.bounds), "tags": src.tags()})
            extracted.append(target)
        mosaic = run.path(f"earth/{name}/native_mosaic.tif")
        mosaic_elevation(extracted, mosaic)
        elevation_with_halo = warp(mosaic, work_grid)
        work_inside = roi_mask(work_grid, area["bbox"])
        assert np.array_equal(work_inside[core], inside)
        features = terrain(elevation_with_halo, SPACING_M, ROUGHNESS_WINDOW, output_mask=work_inside)
        elevation = elevation_with_halo[core].copy()
        elevation[~inside] = np.nan
        del elevation_with_halo, work_inside
        folder = f"earth/{name}/grid_{SPACING_M}m"
        write_raster(run.path(f"{folder}/elevation_m.tif"), elevation, grid, "m", source="FABDEM V1-2")
        complete_features = np.isfinite(elevation)
        for feature, padded_values in features.items():
            values = padded_values[core]
            complete_features &= np.isfinite(values)
            unit = "degree" if feature == "slope_deg" else "1/m" if feature == "laplacian_per_m" else "m"
            write_raster(run.path(f"{folder}/{feature}.tif"), values, grid, unit,
                         spacing_m=SPACING_M, roughness_window=ROUGHNESS_WINDOW)
        del features, padded_values, values
        cover = cover_paths[name]
        run.input(cover)
        run.input(cover.with_suffix(".tif.json"))
        fractions = landcover_fractions(cover, grid)
        for layer, values in fractions.items():
            values[~inside] = np.nan
            write_raster(run.path(f"{folder}/{layer}.tif"), values, grid, "fraction")
        valid = np.isfinite(elevation) & (fractions["source_coverage"] >= 0.99) & inside
        natural = valid & (fractions["built_fraction"] == 0) & (fractions["crop_fraction"] == 0)
        natural &= (fractions["water_fraction"] == 0) & (fractions["bare_sparse_fraction"] >= MIN_BARE_FRACTION)
        write_raster(run.path(f"{folder}/valid_input_mask.tif"), np.where(inside, valid, np.nan), grid, "boolean")
        write_raster(run.path(f"{folder}/natural_screen_mask.tif"), np.where(inside, natural, np.nan), grid, "boolean")
        summary = {"area": name, "grid_crs": str(grid.crs), "transform": list(grid.transform),
                   "shape": [grid.height, grid.width], "source_metadata": source_metadata,
                   "terrain_halo_cells": HALO_CELLS, "processing_shape": [work_grid.height, work_grid.width],
                   "joint_feature_fraction": float(complete_features[inside].mean()),
                   "screened_cells_missing_features": int((natural & ~complete_features).sum()),
                   "elevation_coverage": float(np.isfinite(elevation[inside]).mean()),
                   "valid_input_fraction": float(valid[inside].mean()), "natural_screen_fraction": float(natural[inside].mean())}
        run.path(f"earth/{name}/quality.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        summaries.append(summary)
        print(name, "valid input fraction", summary["valid_input_fraction"])
        del elevation, fractions, valid, natural, inside, complete_features
    run.record["summaries"] = summaries
earth_output = run.directory
print("Completed:", earth_output)


## Inspect before use
Open the elevation, slope and coverage rasters in QGIS or your raster viewer. Check coastlines, tile seams and missing-data holes. Do not treat a completed processing run as scientific approval. Keep spatially separated region/patch groups for later model evaluation; do not randomly split overlapping patches.
